In [1]:
# Rewrite src/preprocessing.py cleanly
preprocessing_code = '''import numpy as np
import joblib
from PIL import Image
from torchvision import transforms

FEATURE_COLS = [
    "soil_pH", "nitrogen", "phosphorus", "potassium",
    "temperature", "humidity", "rainfall",
    "crop_age_days", "sunlight_hours"
]

def get_inference_transform():
    """Image transform for inference — no augmentation."""
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

def preprocess_image(image_path):
    """Load and preprocess a single image for inference."""
    transform = get_inference_transform()
    img = Image.open(image_path).convert("RGB")
    return transform(img).unsqueeze(0)

def preprocess_tabular(features_dict,
                       scaler_path="models/tabular_scaler_v2.pkl"):
    """
    Preprocess tabular features for inference.
    
    Args:
        features_dict: dict with keys matching FEATURE_COLS
        scaler_path: path to fitted StandardScaler
    
    Returns:
        numpy array of shape (1, 9)
    """
    scaler = joblib.load(scaler_path)
    values = np.array([[features_dict[col] for col in FEATURE_COLS]])
    return scaler.transform(values)

def load_label_encoder(path="models/label_encoder_v2.pkl"):
    """Load the tabular label encoder."""
    return joblib.load(path)
'''

with open('src/preprocessing.py', 'w') as f:
    f.write(preprocessing_code)

print("src/preprocessing.py updated!")

src/preprocessing.py updated!


In [2]:
models_code = '''import torch
import torch.nn as nn
import numpy as np
import joblib
from torchvision import models


class FusionMLP(nn.Module):
    """Multimodal fusion MLP combining image and tabular features."""
    def __init__(self, input_dim=2057, num_classes=15):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.network(x)


def load_image_model(weights_path="models/resnet50_balanced.pth",
                     num_classes=15):
    """Load fine-tuned ResNet50 classifier."""
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(2048, num_classes)
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    return model


def get_embedding_model(resnet):
    """Strip final layer to get 2048-d feature extractor."""
    embedding_model = nn.Sequential(*list(resnet.children())[:-1])
    embedding_model.eval()
    return embedding_model


def load_fusion_model(weights_path="models/fusion_mlp_best.pth",
                      input_dim=2057, num_classes=15):
    """Load trained fusion MLP."""
    model = FusionMLP(input_dim=input_dim, num_classes=num_classes)
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    return model


def load_xgboost_model(model_path="models/xgboost_tuned.pkl"):
    """Load tuned XGBoost tabular model."""
    return joblib.load(model_path)


def predict_fusion(image_tensor, tabular_array,
                   embedding_model, fusion_model, image_classes):
    """
    Run full fusion inference.

    Args:
        image_tensor:    preprocessed image tensor (1, 3, 224, 224)
        tabular_array:   scaled tabular features (1, 9)
        embedding_model: ResNet50 feature extractor
        fusion_model:    FusionMLP
        image_classes:   list of class names

    Returns:
        class_name (str), confidence_pct (float)
    """
    with torch.inference_mode():
        embedding = embedding_model(image_tensor)
        embedding = embedding.squeeze(-1).squeeze(-1).numpy()
        fused = np.concatenate([embedding, tabular_array], axis=1)
        fused_tensor = torch.tensor(fused, dtype=torch.float32)
        outputs = fusion_model(fused_tensor)
        probs = torch.softmax(outputs, dim=1)
        confidence, predicted = probs.max(1)

    class_name = image_classes[predicted.item()]
    confidence_pct = round(confidence.item() * 100, 2)
    return class_name, confidence_pct
'''

with open('src/models.py', 'w') as f:
    f.write(models_code)

print("src/models.py updated!")

src/models.py updated!


In [3]:
import subprocess
result = subprocess.run(['pip', 'freeze'], capture_output=True, text=True)
installed = result.stdout

# Key packages we need
key_packages = [
    'torch', 'torchvision', 'fastapi', 'uvicorn', 'streamlit',
    'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'pillow',
    'xgboost', 'joblib', 'requests', 'grad-cam', 'seaborn',
    'pydantic', 'python-multipart', 'opencv-python-headless'
]

requirements = []
for line in installed.split('\n'):
    for pkg in key_packages:
        if line.lower().startswith(pkg.lower()):
            requirements.append(line)
            break

requirements_text = '\n'.join(sorted(requirements))
print("Generated requirements:")
print(requirements_text)

with open('requirements.txt', 'w') as f:
    f.write(requirements_text)
print("\nSaved to requirements.txt!")

Generated requirements:
fastapi==0.136.3
grad-cam==1.5.5
joblib==1.5.3
matplotlib-inline==0.2.2
matplotlib==3.10.9
numpy==2.2.6
pandas==2.3.3
pillow==12.2.0
pydantic==2.13.4
pydantic_core==2.46.4
python-multipart==0.0.32
requests==2.34.2
scikit-learn==1.7.2
seaborn==0.13.2
streamlit==1.58.0
torch==2.12.0
torchaudio==2.11.0
torchvision==0.27.0
uvicorn==0.49.0
xgboost==3.2.0

Saved to requirements.txt!


In [4]:
health_check = '''#!/bin/bash
# Health check script for Crop Disease AI

echo "🌿 Crop Disease AI — Health Check"
echo "=================================="

# Check API
echo "\\n1. Checking API..."
response=$(curl -s -o /dev/null -w "%{http_code}" http://localhost:8000/health)
if [ "$response" = "200" ]; then
    echo "   API: ✓ Running"
else
    echo "   API: ✗ Not running (start with: uvicorn src.api:app --port 8000)"
fi

# Check Streamlit
echo "\\n2. Checking Streamlit..."
response=$(curl -s -o /dev/null -w "%{http_code}" http://localhost:8501)
if [ "$response" = "200" ]; then
    echo "   Streamlit: ✓ Running"
else
    echo "   Streamlit: ✗ Not running (start with: streamlit run app.py)"
fi

# Check model files
echo "\\n3. Checking model files..."
models=("models/resnet50_balanced.pth" "models/fusion_mlp_best.pth" 
        "models/xgboost_tuned.pkl" "models/tabular_scaler_v2.pkl")
for model in "${models[@]}"; do
    if [ -f "$model" ]; then
        echo "   $model: ✓"
    else
        echo "   $model: ✗ MISSING"
    fi
done

echo "\\n=================================="
echo "Health check complete!"
'''

with open('health_check.sh', 'w') as f:
    f.write(health_check)

import os
os.chmod('health_check.sh', 0o755)
print("health_check.sh created!")
print("\nRun it with: bash health_check.sh")

health_check.sh created!

Run it with: bash health_check.sh
